

# Multi-objective Optimization with Optuna

This tutorial showcases Optuna's multi-objective optimization feature by
optimizing the validation accuracy of Fashion MNIST dataset and the FLOPS of the model implemented in PyTorch.

We use [fvcore](https://github.com/facebookresearch/fvcore)_ to measure FLOPS.


In [1]:
%pip install torch torchvision optuna fvcore
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from fvcore.nn import FlopCountAnalysis

import optuna

DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
DIR = ".."
BATCHSIZE = 128
N_TRAIN_EXAMPLES = BATCHSIZE * 30
N_VALID_EXAMPLES = BATCHSIZE * 10


def define_model(trial):
    n_layers = trial.suggest_int("n_layers", 1, 3)
    layers = []

    in_features = 28 * 28
    for i in range(n_layers):
        out_features = trial.suggest_int(f"n_units_l{i}", 4, 128)
        layers.append(nn.Linear(in_features, out_features))
        layers.append(nn.ReLU())
        p = trial.suggest_float(f"dropout_{i}", 0.2, 0.5)
        layers.append(nn.Dropout(p))

        in_features = out_features

    layers.append(nn.Linear(in_features, 10))
    layers.append(nn.LogSoftmax(dim=1))

    return nn.Sequential(*layers)


# Defines training and evaluation.
def train_model(model, optimizer, train_loader):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.view(-1, 28 * 28).to(DEVICE), target.to(DEVICE)
        optimizer.zero_grad()
        F.nll_loss(model(data), target).backward()
        optimizer.step()


def eval_model(model, valid_loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for batch_idx, (data, target) in enumerate(valid_loader):
            data, target = data.view(-1, 28 * 28).to(DEVICE), target.to(DEVICE)
            pred = model(data).argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()

    accuracy = correct / N_VALID_EXAMPLES

    flops = FlopCountAnalysis(model, inputs=(torch.randn(1, 28 * 28).to(DEVICE),)).total()
    return flops, accuracy

Note: you may need to restart the kernel to use updated packages.



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/eugenio/Documents/Notebooks_ArtificialIntelligence/.venv/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/eugenio/Documents/Notebooks_ArtificialIntelligence/.venv/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/Users/eugenio/Documents/Notebooks_ArtificialIntelligence/

Define multi-objective objective function.
Objectives are FLOPS and accuracy.



In [2]:
def objective(trial):
    train_dataset = torchvision.datasets.FashionMNIST(
        DIR, train=True, download=True, transform=torchvision.transforms.ToTensor()
    )
    train_loader = torch.utils.data.DataLoader(
        torch.utils.data.Subset(train_dataset, list(range(N_TRAIN_EXAMPLES))),
        batch_size=BATCHSIZE,
        shuffle=True,
    )

    val_dataset = torchvision.datasets.FashionMNIST(
        DIR, train=False, transform=torchvision.transforms.ToTensor()
    )
    val_loader = torch.utils.data.DataLoader(
        torch.utils.data.Subset(val_dataset, list(range(N_VALID_EXAMPLES))),
        batch_size=BATCHSIZE,
        shuffle=True,
    )
    model = define_model(trial).to(DEVICE)

    optimizer = torch.optim.Adam(
        model.parameters(), trial.suggest_float("lr", 1e-5, 1e-1, log=True)
    )

    for epoch in range(10):
        train_model(model, optimizer, train_loader)
    flops, accuracy = eval_model(model, val_loader)
    return flops, accuracy

## Run multi-objective optimization

If your optimization problem is multi-objective,
Optuna assumes that you will specify the optimization direction for each objective.
Specifically, in this example, we want to minimize the FLOPS (we want a faster model)
and maximize the accuracy. So we set ``directions`` to ``["minimize", "maximize"]``.



In [3]:
study = optuna.create_study(directions=["minimize", "maximize"])
study.optimize(objective, n_trials=30, timeout=300)

print("Number of finished trials: ", len(study.trials))

[I 2026-05-01 22:40:50,283] A new study created in memory with name: no-name-baa7744e-701a-428b-818c-0aff9516db41
[W 2026-05-01 22:40:50,325] Trial 0 failed with parameters: {'n_layers': 3, 'n_units_l0': 88, 'dropout_0': 0.3946915581156233, 'n_units_l1': 35, 'dropout_1': 0.3317001971662271, 'n_units_l2': 88, 'dropout_2': 0.35330971282981205, 'lr': 0.012719132000624927} because of the following error: RuntimeError('Numpy is not available').
Traceback (most recent call last):
  File "/Users/eugenio/Documents/Notebooks_ArtificialIntelligence/.venv/lib/python3.11/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/var/folders/8j/f0xtldm91_1c2bpdr69fxgzw0000gn/T/ipykernel_47233/1864079161.py", line 26, in objective
    train_model(model, optimizer, train_loader)
  File "/var/folders/8j/f0xtldm91_1c2bpdr69fxgzw0000gn/T/ipykernel_47233/304078926.py", line 41, in train_model
    for batch_idx, (data, tar

RuntimeError: Numpy is not available

Note that the following sections requires the installation of [Plotly](https://plotly.com/python)_ for visualization
and [scikit-learn](https://scikit-learn.org/stable)_ for hyperparameter importance calculation:

```console
$ pip install plotly
$ pip install scikit-learn
$ pip install nbformat  # Required if you are running this tutorial in Jupyter Notebook.
```
Check trials on Pareto front visually.



In [ ]:
%pip install plotly scikit-learn nbformat

In [4]:
optuna.visualization.plot_pareto_front(study, target_names=["FLOPS", "accuracy"])

[W 2026-05-01 22:41:50,346] Your study does not have any completed trials. 


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'hovertemplate': '%{text}<extra>Trial</extra>',
              'marker': {'color': [],
                         'colorbar': {'title': {'text': 'Trial'}},
                         'colorscale': [[0.0, 'rgb(247,251,255)'], [0.125,
                                        'rgb(222,235,247)'], [0.25,
                                        'rgb(198,219,239)'], [0.375,
                                        'rgb(158,202,225)'], [0.5,
                                        'rgb(107,174,214)'], [0.625,
                                        'rgb(66,146,198)'], [0.75,
                                        'rgb(33,113,181)'], [0.875,
                                        'rgb(8,81,156)'], [1.0, 'rgb(8,48,107)']],
                         'line': {'color': 'Grey', 'width': 0.5}},
              'mode': 'markers',
              'showlegend': False,
              'text': [],
              'type': 'scatter',
              'x': [],
              'y': []},
             {'hovertemplate': '%{text}<extra>Best Trial</extra>',
              'marker': {'color': [],
                         'colorbar': {'title': {'text': 'Best Trial'}, 'x': 1.1, 'xpad': 40},
                         'colorscale': [[0.0, 'rgb(255,245,240)'], [0.125,
                                        'rgb(254,224,210)'], [0.25,
                                        'rgb(252,187,161)'], [0.375,
                                        'rgb(252,146,114)'], [0.5,
                                        'rgb(251,106,74)'], [0.625,
                                        'rgb(239,59,44)'], [0.75,
                                        'rgb(203,24,29)'], [0.875,
                                        'rgb(165,15,21)'], [1.0, 'rgb(103,0,13)']],
                         'line': {'color': 'Grey', 'width': 0.5}},
              'mode': 'markers',
              'showlegend': False,
              'text': [],
              'type': 'scatter',
              'x': [],
              'y': []}],
    'layout': {'template': '...',
               'title': {'text': 'Pareto-front Plot'},
               'xaxis': {'title': {'text': 'FLOPS'}},
               'yaxis': {'title': {'text': 'accuracy'}}}
})

Fetch the list of trials on the Pareto front with :attr:`~optuna.study.Study.best_trials`.

For example, the following code shows the number of trials on the Pareto front and picks the trial with the highest accuracy.



In [ ]:
print(f"Number of trials on the Pareto front: {len(study.best_trials)}")

trial_with_highest_accuracy = max(study.best_trials, key=lambda t: t.values[1])
print("Trial with highest accuracy: ")
print(f"\tnumber: {trial_with_highest_accuracy.number}")
print(f"\tparams: {trial_with_highest_accuracy.params}")
print(f"\tvalues: {trial_with_highest_accuracy.values}")

Learn which hyperparameters are affecting the flops most with hyperparameter importance.



In [ ]:
optuna.visualization.plot_param_importances(
    study, target=lambda t: t.values[0], target_name="flops"
)